In [3]:
# ============================================================
# CELL 1: Install libraries + imports
# Run this first. Wait for it to finish before moving to Cell 2.
# ============================================================

# Install one library that Colab doesn't have by default
!pip install -q plotly

# --- Standard imports ---
import pandas as pd
import sqlite3
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# --- Display settings ---
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.2f}".format)

print("✅ All libraries loaded. Move to Cell 2.")

✅ All libraries loaded. Move to Cell 2.


In [22]:
# ============================================================
# CELL 2: Upload & Load CSV
# ============================================================
from google.colab import files

print("A file picker will appear. Select your perfume_prices CSV file.")
uploaded = files.upload()
# files.upload() opens a file picker in your browser
# Navigate to your project folder and select the CSV

import io
filename = list(uploaded.keys())[0]
# list(uploaded.keys())[0] = gets the name of the file you uploaded

df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))
# BytesIO converts the uploaded file bytes into something pandas can read

print(f"✅ Loaded {len(df_raw)} rows, {df_raw.shape[1]} columns.")
print(df_raw.head())
print(df_raw["platform"].value_counts())
print(f"\nRows with price: {df_raw['price_inr'].notna().sum()}")
print(f"Rows without price: {df_raw['price_inr'].isna().sum()}")
print(f"\nPrice sample:\n{df_raw['price_inr'].head(10)}")

A file picker will appear. Select your perfume_prices CSV file.


Saving perfume_prices_raw.csv to perfume_prices_raw (1).csv
✅ Loaded 5254 rows, 11 columns.
    platform search_term                                       product_name  \
0  Amazon.in     perfume  Forest Essentials Travel Size Fragrant Body Mi...   
1  Amazon.in     perfume  HIRA After Hours Perfume for Men | Perfume for...   
2  Amazon.in     perfume  Fastrack Men Perfume Wood Scent Spray Trance, ...   
3  Amazon.in     perfume  BELLAVITA Magical Scents | 4x20ml | Long Lasti...   
4  Amazon.in     perfume  Wild Stone Edge Edp Premium Perfume For Men,10...   

       brand  price_inr  mrp_inr  discount_pct price_tier  rating  \
0     Forest   1,495.00 1,495.00          0.00  Mid-range     NaN   
1       HIRA     899.00      NaN           NaN  Mid-range     NaN   
2   Fastrack     545.00      NaN           NaN  Mid-range     NaN   
3  BELLAVITA     449.00      NaN           NaN     Budget     NaN   
4       Wild     384.00      NaN           NaN     Budget     NaN   

   review_count   

In [27]:
# ============================================================
# CELL 3: Clean the Data — Updated with outlier removal
# ============================================================

df = df_raw.copy()
df["price_inr"] = df["price_inr"].astype(str).str.replace(",", "").astype(float)
df["mrp_inr"]   = df["mrp_inr"].astype(str).str.replace(",", "").astype(float)

# 1. Strip whitespace from text columns
for col in ["platform", "product_name", "brand"]:
    df[col] = df[col].astype(str).str.strip()

# 2. Convert price columns to numeric
df["price_inr"] = pd.to_numeric(df["price_inr"], errors="coerce")
df["mrp_inr"]   = pd.to_numeric(df["mrp_inr"],   errors="coerce")

# 3. Drop rows with no product name
df = df.dropna(subset=["product_name"])
df = df[df["product_name"] != "N/A"]
df = df[df["product_name"].str.len() > 5]
print(f"After name filter: {len(df)} rows")

# 4. Remove price outliers
# Realistic Indian perfume range: ₹50 → ₹50,000
df = df[
    df["price_inr"].isna() |                          # keep rows with no price
    df["price_inr"].between(50, 50000)               # keep realistic prices
]
print(f"After price range filter: {len(df)} rows")

# 5. Remove parsing artifacts — prices with 4+ trailing zeros
# These are concatenation bugs e.g. 575100042
mask_artifact = (
    df["price_inr"].notna() &
    df["price_inr"].astype(str).str.match(r"^\d+0{4,}\.0$")
)
df = df[~mask_artifact]
print(f"After artifact filter: {len(df)} rows")

# 6. Remove junk product names
junk_keywords = ["buy ", "online", "at best price", "at amazing", "click here"]
mask_junk = df["product_name"].str.lower().apply(
    lambda x: any(k in x for k in junk_keywords)
)
df = df[~mask_junk]
print(f"After junk name filter: {len(df)} rows")

# 7. Add price tier
def price_tier(price):
    if pd.isna(price):    return "Unknown"
    elif price < 500:     return "Budget"
    elif price < 2000:    return "Mid-range"
    elif price < 6000:    return "Premium"
    else:                 return "Luxury"

df["price_tier"] = df["price_inr"].apply(price_tier)

# 8. Clean rating column
df["rating"] = pd.to_numeric(
    df["rating"].astype(str).str.extract(r"(\d+\.?\d*)")[0],
    errors="coerce"
)
# Remove impossible ratings
df.loc[~df["rating"].between(1, 5), "rating"] = None

# 9. Parse scraped_at
df["scraped_at"] = pd.to_datetime(df["scraped_at"], errors="coerce", utc=True)

# 10. Drop duplicates
before = len(df)
df.drop_duplicates(subset=["platform", "product_name", "price_inr"], inplace=True)
print(f"After dedup: {len(df)} rows (removed {before - len(df)} duplicates)")

# 11. Final summary
print(f"\n✅ Clean dataset: {len(df)} rows")
print(f"\nPlatform breakdown:\n{df['platform'].value_counts()}")
print(f"\nPrice tier breakdown:\n{df['price_tier'].value_counts()}")
print(f"\nPrice stats (priced rows only):")
print(df[df["price_inr"].notna()]["price_inr"].describe().round(0))
print(f"\nMissing prices: {df['price_inr'].isna().sum()} rows")

# 12. Download clean file
from google.colab import files
df.to_csv("perfume_prices_clean.csv", index=False, encoding="utf-8-sig")
files.download("perfume_prices_clean.csv")

After name filter: 5223 rows
After price range filter: 5213 rows
After artifact filter: 5213 rows
After junk name filter: 5208 rows
After dedup: 2067 rows (removed 3141 duplicates)

✅ Clean dataset: 2067 rows

Platform breakdown:
platform
Amazon.in    1124
Flipkart      783
Nykaa         160
Name: count, dtype: int64

Price tier breakdown:
price_tier
Budget       1358
Mid-range     547
Premium        78
Luxury         57
Unknown        27
Name: count, dtype: int64

Price stats (priced rows only):
count    2,040.00
mean       890.00
std      2,401.00
min         57.00
25%        229.00
50%        364.00
75%        649.00
max     48,496.00
Name: price_inr, dtype: float64

Missing prices: 27 rows


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# ============================================================
# CELL 4: Load into SQLite (your in-notebook warehouse)
# SQLite = a lightweight database that lives inside your notebook.
# Think of it as BigQuery but running locally for free.
# ============================================================

conn = sqlite3.connect(":memory:")
# ":memory:" = database lives in RAM, gone when notebook closes
# Perfect for analysis — no files needed

df.to_sql("perfume_prices", conn, if_exists="replace", index=False)
# Loads the entire DataFrame into a SQL table called "perfume_prices"

def sql(query):
    """Helper: run any SQL query and return a DataFrame."""
    return pd.read_sql_query(query, conn)

print("✅ Data loaded into SQLite. You can now run SQL queries.")
print(sql("SELECT platform, COUNT(*) as products FROM perfume_prices GROUP BY platform"))


✅ Data loaded into SQLite. You can now run SQL queries.
    platform  products
0  Amazon.in      1131
1   Flipkart       783
2      Nykaa       160


In [7]:

# ============================================================
# CELL 5: ANALYSIS 1 — Platform Price Comparison
# Business question: Which platform is cheapest for perfume?
# ============================================================

platform_stats = sql("""
    SELECT
        platform,
        COUNT(*)                        AS total_products,
        ROUND(AVG(price_inr), 0)        AS avg_price,
        ROUND(MIN(price_inr), 0)        AS min_price,
        ROUND(MAX(price_inr), 0)        AS max_price,
        ROUND(AVG(discount_pct), 1)     AS avg_discount_pct
    FROM perfume_prices
    WHERE price_inr IS NOT NULL
    GROUP BY platform
    ORDER BY avg_price
""")

print("=== PLATFORM PRICE COMPARISON ===")
print(platform_stats.to_string(index=False))

fig1 = px.bar(
    platform_stats,
    x="platform",
    y="avg_price",
    color="platform",
    text="avg_price",
    title="Average Perfume Price by Platform (₹)",
    labels={"avg_price": "Avg Price (₹)", "platform": "Platform"},
    color_discrete_map={
        "Amazon.in": "#FF9900",
        "Flipkart":  "#2874F0",
        "Nykaa":     "#FC2779"
    }
)
fig1.update_traces(texttemplate="₹%{text:,.0f}", textposition="outside")
fig1.update_layout(showlegend=False, plot_bgcolor="white")
fig1.show()



=== PLATFORM PRICE COMPARISON ===
 platform  total_products  avg_price  min_price  max_price  avg_discount_pct
 Flipkart             783     458.00      57.00   7,800.00               NaN
Amazon.in            1104   1,333.00      59.00 114,096.00           -784.60
    Nykaa             160   3,342.00      70.00  34,000.00               NaN


In [8]:

# ============================================================
# CELL 6: ANALYSIS 2 — Price Tier Distribution per Platform
# Business question: Does each platform target different customer segments?
# ============================================================

tier_dist = sql("""
    SELECT
        platform,
        price_tier,
        COUNT(*) AS product_count
    FROM perfume_prices
    WHERE price_tier != 'Unknown'
    GROUP BY platform, price_tier
    ORDER BY platform, price_tier
""")

print("=== PRICE TIER DISTRIBUTION ===")
print(tier_dist.to_string(index=False))

fig2 = px.bar(
    tier_dist,
    x="platform",
    y="product_count",
    color="price_tier",
    barmode="group",
    title="Price Tier Distribution by Platform",
    labels={"product_count": "Number of Products", "price_tier": "Tier"},
    category_orders={"price_tier": ["Budget", "Mid-range", "Premium", "Luxury"]},
    color_discrete_map={
        "Budget":    "#2ecc71",
        "Mid-range": "#3498db",
        "Premium":   "#9b59b6",
        "Luxury":    "#e74c3c"
    }
)
fig2.update_layout(plot_bgcolor="white")
fig2.show()



=== PRICE TIER DISTRIBUTION ===
 platform price_tier  product_count
Amazon.in     Budget            730
Amazon.in     Luxury             25
Amazon.in  Mid-range            302
Amazon.in    Premium             47
 Flipkart     Budget            593
 Flipkart     Luxury              2
 Flipkart  Mid-range            184
 Flipkart    Premium              4
    Nykaa     Budget             35
    Nykaa     Luxury             37
    Nykaa  Mid-range             61
    Nykaa    Premium             27


In [9]:

# ============================================================
# CELL 7: ANALYSIS 3 — Top 15 Most Expensive Products
# Business question: What are the luxury anchors in this market?
# ============================================================

top_expensive = sql("""
    SELECT
        platform,
        product_name,
        price_inr,
        price_tier
    FROM perfume_prices
    WHERE price_inr IS NOT NULL
    ORDER BY price_inr DESC
    LIMIT 15
""")

print("=== TOP 15 MOST EXPENSIVE PRODUCTS ===")
print(top_expensive.to_string(index=False))

fig3 = px.bar(
    top_expensive,
    x="price_inr",
    y="product_name",
    color="platform",
    orientation="h",
    title="Top 15 Most Expensive Perfumes (₹)",
    labels={"price_inr": "Price (₹)", "product_name": "Product"},
    color_discrete_map={
        "Amazon.in": "#FF9900",
        "Flipkart":  "#2874F0",
        "Nykaa":     "#FC2779"
    }
)
fig3.update_layout(yaxis={"categoryorder": "total ascending"}, plot_bgcolor="white")
fig3.show()



=== TOP 15 MOST EXPENSIVE PRODUCTS ===
 platform                                                                              product_name  price_inr price_tier
Amazon.in                                 ROZARK Long Lasting Fresh Rose sat Fragrance Perfume-9 ml 114,096.00     Luxury
Amazon.in                                               Clive Christian X Perfume Spray 100ml/3.4oz  89,000.00     Luxury
Amazon.in                                                  Clive Christian 1872 Perfume Spray 100Ml  84,500.00     Luxury
Amazon.in                                                      Bond No 9 Colognes Queens, 3.4 Ounce  75,833.00     Luxury
Amazon.in                   Tom Ford Private Blend Oud Wood Intense Eau De Parfum Spray 100ml/3.3oz  67,399.00     Luxury
Amazon.in                                                             1872 Perfume Spray 50ml/1.6oz  63,000.00     Luxury
Amazon.in                                            Creed Royal Mayfair 2.5 oz Eau de Parfum Spray  54,765

In [10]:

# ============================================================
# CELL 8: ANALYSIS 4 — Budget Products (under ₹500)
# Business question: Which platform has the most affordable options?
# ============================================================

budget = sql("""
    SELECT
        platform,
        COUNT(*) AS budget_products,
        ROUND(AVG(price_inr), 0) AS avg_budget_price
    FROM perfume_prices
    WHERE price_inr < 500
    GROUP BY platform
    ORDER BY budget_products DESC
""")

print("=== BUDGET PRODUCTS (under ₹500) ===")
print(budget.to_string(index=False))

fig4 = px.pie(
    budget,
    names="platform",
    values="budget_products",
    title="Share of Budget Perfumes (under ₹500) by Platform",
    color="platform",
    color_discrete_map={
        "Amazon.in": "#FF9900",
        "Flipkart":  "#2874F0",
        "Nykaa":     "#FC2779"
    }
)
fig4.show()


=== BUDGET PRODUCTS (under ₹500) ===
 platform  budget_products  avg_budget_price
Amazon.in              730            281.00
 Flipkart              593            278.00
    Nykaa               35            318.00


In [11]:

# ============================================================
# CELL 9: ANALYSIS 5 — Price Distribution (Box Plot)
# Business question: How spread out are prices on each platform?
# ============================================================

df_priced = df[df["price_inr"].notna()].copy()

fig5 = px.box(
    df_priced,
    x="platform",
    y="price_inr",
    color="platform",
    title="Price Distribution by Platform (₹)",
    labels={"price_inr": "Price (₹)", "platform": "Platform"},
    color_discrete_map={
        "Amazon.in": "#FF9900",
        "Flipkart":  "#2874F0",
        "Nykaa":     "#FC2779"
    }
)
fig5.update_layout(showlegend=False, plot_bgcolor="white", yaxis_type="log")
# yaxis_type="log" = log scale because prices range from ₹99 to ₹10,000+
fig5.show()



In [12]:

# ============================================================
# CELL 10: ANALYSIS 6 — Search Term Performance
# Business question: Which search term yields the most products?
# ============================================================

term_perf = sql("""
    SELECT
        search_term,
        platform,
        COUNT(*)                    AS products_found,
        ROUND(AVG(price_inr), 0)   AS avg_price
    FROM perfume_prices
    WHERE price_inr IS NOT NULL
    GROUP BY search_term, platform
    ORDER BY search_term, platform
""")

print("=== SEARCH TERM PERFORMANCE ===")
print(term_perf.to_string(index=False))

fig6 = px.bar(
    term_perf,
    x="search_term",
    y="products_found",
    color="platform",
    barmode="group",
    title="Products Found per Search Term by Platform",
    labels={"products_found": "Products Found", "search_term": "Search Term"},
    color_discrete_map={
        "Amazon.in": "#FF9900",
        "Flipkart":  "#2874F0",
        "Nykaa":     "#FC2779"
    }
)
fig6.update_layout(plot_bgcolor="white")
fig6.show()



=== SEARCH TERM PERFORMANCE ===
      search_term  platform  products_found  avg_price
   Indian perfume Amazon.in             114     670.00
   Indian perfume  Flipkart              63     403.00
   Indian perfume     Nykaa              20   5,746.00
            attar Amazon.in             145     496.00
            attar  Flipkart              83     324.00
            attar     Nykaa              20     644.00
        body mist Amazon.in              82     500.00
        body mist  Flipkart              85     777.00
        body mist     Nykaa              13   1,493.00
   budget perfume Amazon.in              82     411.00
   budget perfume  Flipkart              78     294.00
   budget perfume     Nykaa              18   1,546.00
          cologne Amazon.in             112   1,983.00
          cologne  Flipkart              95     568.00
          cologne     Nykaa              18   6,634.00
deodorant perfume Amazon.in             133     298.00
deodorant perfume  Flipkart      

In [13]:

# ============================================================
# CELL 11: BUSINESS SUMMARY — what you say in an interview
# ============================================================

summary = sql("""
    SELECT
        platform,
        COUNT(*)                        AS total_listings,
        ROUND(AVG(price_inr), 0)        AS avg_price_inr,
        ROUND(MIN(price_inr), 0)        AS min_price_inr,
        ROUND(MAX(price_inr), 0)        AS max_price_inr,
        SUM(CASE WHEN price_tier='Budget'    THEN 1 ELSE 0 END) AS budget_count,
        SUM(CASE WHEN price_tier='Mid-range' THEN 1 ELSE 0 END) AS midrange_count,
        SUM(CASE WHEN price_tier='Premium'   THEN 1 ELSE 0 END) AS premium_count,
        SUM(CASE WHEN price_tier='Luxury'    THEN 1 ELSE 0 END) AS luxury_count
    FROM perfume_prices
    WHERE price_inr IS NOT NULL
    GROUP BY platform
    ORDER BY avg_price_inr
""")

print("=" * 60)
print("BUSINESS SUMMARY — INDIAN PERFUME PRICING INTELLIGENCE")
print("=" * 60)
print(summary.to_string(index=False))
print()
print("KEY INSIGHTS FOR INTERVIEW:")
print("-" * 60)

for _, row in summary.iterrows():
    print(f"""
Platform : {row['platform']}
Listings : {int(row['total_listings'])} products analysed
Avg Price: ₹{int(row['avg_price_inr']):,}
Range    : ₹{int(row['min_price_inr']):,} → ₹{int(row['max_price_inr']):,}
Segments : Budget={int(row['budget_count'])} | Mid={int(row['midrange_count'])} | Premium={int(row['premium_count'])} | Luxury={int(row['luxury_count'])}
""")

print("✅ Full analysis complete. Save this notebook and upload to GitHub.")

BUSINESS SUMMARY — INDIAN PERFUME PRICING INTELLIGENCE
 platform  total_listings  avg_price_inr  min_price_inr  max_price_inr  budget_count  midrange_count  premium_count  luxury_count
 Flipkart             783         458.00          57.00       7,800.00           593             184              4             2
Amazon.in            1104       1,333.00          59.00     114,096.00           730             302             47            25
    Nykaa             160       3,342.00          70.00      34,000.00            35              61             27            37

KEY INSIGHTS FOR INTERVIEW:
------------------------------------------------------------

Platform : Flipkart
Listings : 783 products analysed
Avg Price: ₹458
Range    : ₹57 → ₹7,800
Segments : Budget=593 | Mid=184 | Premium=4 | Luxury=2


Platform : Amazon.in
Listings : 1104 products analysed
Avg Price: ₹1,333
Range    : ₹59 → ₹114,096
Segments : Budget=730 | Mid=302 | Premium=47 | Luxury=25


Platform : Nykaa
Listings 

In [14]:
# ============================================================
# CELL 12: ANALYSIS 7 — Brand-Level Pricing Intelligence
# Business question: Which brands dominate each platform
# and how does their pricing differ across platforms?
# ============================================================

# STEP 1: Extract brand name from product_name
# Most product names start with the brand — we grab the first word
# Example: "Bella Vita Luxury Perfumes..." → "Bella Vita"

df["brand_clean"] = df["product_name"].str.extract(r"^([A-Za-z]+(?:\s[A-Za-z]+)?)")
# r"^([A-Za-z]+(?:\s[A-Za-z]+)?)" = regex that grabs the first 1-2 words
# ^ = start of string
# [A-Za-z]+ = one or more letters
# (?:\s[A-Za-z]+)? = optionally grab a second word too

# Push updated df back into SQLite
df.to_sql("perfume_prices", conn, if_exists="replace", index=False)

# STEP 2: Top brands by listing count
brand_volume = sql("""
    SELECT
        brand_clean                         AS brand,
        COUNT(*)                            AS total_listings,
        COUNT(DISTINCT platform)            AS platforms_present,
        ROUND(AVG(price_inr), 0)            AS avg_price,
        ROUND(MIN(price_inr), 0)            AS min_price,
        ROUND(MAX(price_inr), 0)            AS max_price
    FROM perfume_prices
    WHERE price_inr IS NOT NULL
      AND brand_clean IS NOT NULL
    GROUP BY brand_clean
    HAVING total_listings >= 2             -- only brands with 2+ listings
    ORDER BY total_listings DESC
    LIMIT 15
""")
# HAVING = filters after GROUP BY (like WHERE but for aggregated results)
# total_listings >= 2 = ignore one-off products, focus on real brands

print("=== TOP 15 BRANDS BY LISTING VOLUME ===")
print(brand_volume.to_string(index=False))

# STEP 3: Brands present on multiple platforms (cross-platform brands)
cross_platform = sql("""
    SELECT
        brand_clean                         AS brand,
        COUNT(DISTINCT platform)            AS platforms,
        GROUP_CONCAT(DISTINCT platform)     AS platform_list,
        ROUND(AVG(price_inr), 0)            AS avg_price
    FROM perfume_prices
    WHERE price_inr IS NOT NULL
      AND brand_clean IS NOT NULL
    GROUP BY brand_clean
    HAVING platforms > 1
    ORDER BY platforms DESC, avg_price DESC
""")
# GROUP_CONCAT = joins values into one string e.g. "Amazon.in,Nykaa"

print("\n=== BRANDS SELLING ON MULTIPLE PLATFORMS ===")
print(cross_platform.to_string(index=False))

# STEP 4: Platform pricing rank per brand (window function)
brand_platform_rank = sql("""
    SELECT
        brand_clean                                         AS brand,
        platform,
        ROUND(AVG(price_inr), 0)                           AS avg_price,
        RANK() OVER (
            PARTITION BY brand_clean
            ORDER BY AVG(price_inr) DESC
        )                                                   AS price_rank
    FROM perfume_prices
    WHERE price_inr IS NOT NULL
      AND brand_clean IS NOT NULL
    GROUP BY brand_clean, platform
    HAVING COUNT(*) >= 2
    ORDER BY brand, price_rank
""")
# RANK() OVER (PARTITION BY brand ORDER BY avg_price DESC)
# = for each brand separately, rank platforms from most expensive to cheapest
# This is a WINDOW FUNCTION — mention this in your interview

print("\n=== PLATFORM PRICE RANK PER BRAND ===")
print(brand_platform_rank.to_string(index=False))

# STEP 5: Visualise top 10 brands by avg price
top_brands = brand_volume.head(10)

fig7 = px.bar(
    top_brands,
    x="brand",
    y="avg_price",
    color="platforms_present",
    text="avg_price",
    title="Top 10 Brands — Avg Price & Platform Reach",
    labels={
        "avg_price": "Avg Price (₹)",
        "brand": "Brand",
        "platforms_present": "# Platforms"
    },
    color_continuous_scale="Viridis"
)
fig7.update_traces(texttemplate="₹%{text:,.0f}", textposition="outside")
fig7.update_layout(plot_bgcolor="white", xaxis_tickangle=-30)
fig7.show()

# STEP 6: Interview-ready brand insight
print("\n" + "=" * 60)
print("BRAND INSIGHTS FOR INTERVIEW")
print("=" * 60)
print(f"Total unique brands identified : {df['brand_clean'].nunique()}")
print(f"Brands on 2+ platforms         : {len(cross_platform)}")
if len(cross_platform) > 0:
    top = brand_volume.iloc[0]
    print(f"Most listed brand              : {top['brand']} ({int(top['total_listings'])} listings)")
    print(f"Highest avg price brand        : {brand_volume.sort_values('avg_price', ascending=False).iloc[0]['brand']}")
print("\n✅ Cell 12 complete. Your analysis is done.")

=== TOP 15 BRANDS BY LISTING VOLUME ===
           brand  total_listings  platforms_present  avg_price  min_price  max_price
      Wild Stone              62                  2     334.00     166.00     880.00
      Bella Vita              49                  2     445.00      84.00   1,449.00
  Yardley London              39                  2     286.00      70.00     559.00
     Park Avenue              30                  1     322.00     172.00     720.00
            BATH              22                  1   1,428.00     683.00   4,392.00
  Plum BodyLovin              20                  3     456.00     244.00     960.00
         THE MAN              19                  1     446.00     249.00   1,499.00
       La French              19                  2     313.00     179.00     463.00
        Skinn By              17                  2   1,089.00     497.00   2,795.00
           Layer              17                  2     218.00     111.00     429.00
         The Man         


BRAND INSIGHTS FOR INTERVIEW
Total unique brands identified : 1048
Brands on 2+ platforms         : 86
Most listed brand              : Wild Stone (62 listings)
Highest avg price brand        : Jo Malone

✅ Cell 12 complete. Your analysis is done.


In [37]:
# ============================================================
# CELL 13: Product Matching + Arbitrage Detection
# Uses fuzzy string matching to find same product across platforms
# Then identifies price differences = arbitrage opportunities
# ============================================================

# Install rapidfuzz if not already installed
!pip install -q rapidfuzz

from rapidfuzz import fuzz, process
import pandas as pd
import plotly.express as px
import itertools

# ============================================================
# STEP 1: Load and prep data
# ============================================================

# If running fresh, upload your new CSV first
from google.colab import files
uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])

# If continuing from previous cells, df is already loaded
# Filter to only priced rows
df_priced = df[df["price_inr"].notna()].copy()
df_priced = df_priced[df_priced["price_inr"] > 0]
df_priced["product_name"] = df_priced["product_name"].astype(str).str.strip()

print(f"Working with {len(df_priced)} priced rows across platforms:")
print(df_priced["platform"].value_counts())

# ============================================================
# STEP 2: Fuzzy Product Matching
# rapidfuzz compares two product names and returns a score
# Score 100 = identical, 0 = completely different
# We use 80+ as "same product"
# ============================================================

MATCH_THRESHOLD = 80
# 80 means: "at least 80% similar in words"
# Lower = more matches but more false positives
# Higher = fewer matches but more accurate

def normalize_name(name):
    """
    Cleans product name before matching.
    Removes ml sizes, common noise words, extra spaces.
    Example: "Dior Sauvage EDT 100ml" → "dior sauvage edt"
    """
    name = str(name).lower()
    name = re.sub(r"\d+\s*ml", "", name)      # remove "100ml", "50 ml"
    name = re.sub(r"\d+\s*gm", "", name)      # remove "200gm"
    name = re.sub(r"[^\w\s]", " ", name)      # remove punctuation
    name = re.sub(r"\s+", " ", name).strip()  # clean extra spaces
    return name

df_priced["name_normalized"] = df_priced["product_name"].apply(normalize_name)

# ============================================================
# STEP 3: Find matching products across platforms
# For each platform pair, find products with similar names
# ============================================================

platforms = df_priced["platform"].unique().tolist()
print(f"\nPlatforms to compare: {platforms}")

matches = []

# Compare every platform pair
# itertools.combinations gives us: (Amazon, Flipkart), (Amazon, Nykaa), (Flipkart, Nykaa)
for platform_a, platform_b in itertools.combinations(platforms, 2):

    df_a = df_priced[df_priced["platform"] == platform_a].copy()
    df_b = df_priced[df_priced["platform"] == platform_b].copy()

    print(f"\nMatching {platform_a} ({len(df_a)}) vs {platform_b} ({len(df_b)})...")

    # Limit to 500 products per platform to avoid very long runtime
    df_a_sample = df_a.head(500)
    df_b_sample = df_b.head(500)

    names_b = df_b_sample["name_normalized"].tolist()

    for _, row_a in df_a_sample.iterrows():
        # process.extractOne = finds the best match in names_b for row_a's name
        # Returns: (matched_name, score, index)
        result = process.extractOne(
            row_a["name_normalized"],
            names_b,
            scorer=fuzz.token_sort_ratio
            # token_sort_ratio = sorts words before comparing
            # so "Dior Sauvage EDT" matches "Sauvage EDT Dior"
        )

        if result and result[1] >= MATCH_THRESHOLD:
            matched_name, score, idx = result
            row_b = df_b_sample.iloc[idx]

            price_a = row_a["price_inr"]
            price_b = row_b["price_inr"]
            price_diff = abs(price_a - price_b)
            price_diff_pct = round((price_diff / max(price_a, price_b)) * 100, 1)

            # Only keep matches with meaningful price difference (>5%)
            if price_diff_pct > 5:
                cheaper_platform = platform_a if price_a < price_b else platform_b
                dearer_platform  = platform_b if price_a < price_b else platform_a
                cheaper_price    = min(price_a, price_b)
                dearer_price     = max(price_a, price_b)

                matches.append({
                    "product_a":        row_a["product_name"],
                    "product_b":        row_b["product_name"],
                    "match_score":      score,
                    "platform_a":       platform_a,
                    "platform_b":       platform_b,
                    "price_a":          price_a,
                    "price_b":          price_b,
                    "cheaper_platform": cheaper_platform,
                    "dearer_platform":  dearer_platform,
                    "cheaper_price":    cheaper_price,
                    "dearer_price":     dearer_price,
                    "price_diff_inr":   round(price_diff, 0),
                    "price_diff_pct":   price_diff_pct
                })

df_matches = pd.DataFrame(matches)

if df_matches.empty:
    print("\n⚠️ No matches found. Try lowering MATCH_THRESHOLD to 70.")
else:
    print(f"\n✅ Found {len(df_matches)} cross-platform product matches")
    print(df_matches[[
        "product_a", "platform_a", "price_a",
        "platform_b", "price_b", "price_diff_pct", "cheaper_platform"
    ]].head(10).to_string(index=False))

# ============================================================
# STEP 4: Arbitrage Table
# Same product, different price = arbitrage opportunity
# A reseller could buy from cheaper platform, sell on dearer one
# ============================================================

if not df_matches.empty:
    # Sort by biggest price gap
    arbitrage = df_matches.sort_values("price_diff_inr", ascending=False)

    print("\n=== TOP 20 ARBITRAGE OPPORTUNITIES ===")
    print(f"{'Product':<50} {'Buy From':<12} {'At':<8} {'Sell On':<12} {'At':<8} {'Profit'}")
    print("-" * 105)
    for _, row in arbitrage.head(20).iterrows():
        name = str(row["product_a"])[:48]
        print(
            f"{name:<50} {row['cheaper_platform']:<12} "
            f"₹{int(row['cheaper_price']):<7} {row['dearer_platform']:<12} "
            f"₹{int(row['dearer_price']):<7} ₹{int(row['price_diff_inr'])}"
        )

    # ============================================================
    # STEP 5: Visualise arbitrage opportunities
    # ============================================================

    top_arb = arbitrage.head(15).copy()
    top_arb["label"] = top_arb["product_a"].str[:30]

    fig = px.bar(
        top_arb,
        x="price_diff_inr",
        y="label",
        color="cheaper_platform",
        orientation="h",
        title="Top 15 Arbitrage Opportunities — Price Gap (₹)",
        labels={
            "price_diff_inr": "Price Difference (₹)",
            "label": "Product",
            "cheaper_platform": "Cheaper On"
        },
        color_discrete_map={
            "Amazon.in": "#FF9900",
            "Flipkart":  "#2874F0",
            "Nykaa":     "#FC2779"
        }
    )
    fig.update_layout(
        yaxis={"categoryorder": "total ascending"},
        plot_bgcolor="white"
    )
    fig.show()

    # ============================================================
    # STEP 6: Summary stats for interview
    # ============================================================

    print("\n" + "=" * 60)
    print("ARBITRAGE SUMMARY — FOR INTERVIEW")
    print("=" * 60)
    print(f"Total cross-platform matches found : {len(df_matches)}")
    print(f"Avg price gap                      : ₹{df_matches['price_diff_inr'].mean():.0f}")
    print(f"Max price gap                      : ₹{df_matches['price_diff_inr'].max():.0f}")
    print(f"Products cheaper on Amazon         : {(df_matches['cheaper_platform']=='Amazon.in').sum()}")
    print(f"Products cheaper on Flipkart       : {(df_matches['cheaper_platform']=='Flipkart').sum()}")
    print(f"Products cheaper on Nykaa          : {(df_matches['cheaper_platform']=='Nykaa').sum()}")
    print(f"\n✅ Cell 13 complete.")

    # Save matches to CSV for dashboard
    df_matches.to_csv("perfume_arbitrage.csv", index=False)
    print("💾 Saved arbitrage table to perfume_arbitrage.csv")

Saving perfume_prices_clean.csv to perfume_prices_clean (2).csv
Working with 2040 priced rows across platforms:
platform
Amazon.in    1097
Flipkart      783
Nykaa         160
Name: count, dtype: int64

Platforms to compare: ['Amazon.in', 'Flipkart', 'Nykaa']

Matching Amazon.in (1097) vs Flipkart (783)...

Matching Amazon.in (1097) vs Nykaa (160)...

Matching Flipkart (783) vs Nykaa (160)...

✅ Found 13 cross-platform product matches
                                                                                                      product_a platform_a  price_a platform_b  price_b  price_diff_pct cheaper_platform
                                            Wild Stone Edge Eau De Parfum for Men, Long Lasting perfume, 100 ml  Amazon.in   384.00   Flipkart   281.00           26.80         Flipkart
                                                                  Wild Stone Hydra Energy Perfume for Men, 50ml  Amazon.in   233.00   Flipkart   176.00           24.50         Flipkart
       


ARBITRAGE SUMMARY — FOR INTERVIEW
Total cross-platform matches found : 13
Avg price gap                      : ₹1143
Max price gap                      : ₹7001
Products cheaper on Amazon         : 6
Products cheaper on Flipkart       : 6
Products cheaper on Nykaa          : 1

✅ Cell 13 complete.
💾 Saved arbitrage table to perfume_arbitrage.csv


In [38]:
# ============================================================
# CELL 14: Price Recommendation Engine
# SQL-based pricing recommendations per product
# Defensible logic, no fake ML
# ============================================================

# ============================================================
# STEP 1: Reload cleaned data into SQLite
# (in case conn was reset)
# ============================================================

import sqlite3
import pandas as pd
import plotly.express as px

conn = sqlite3.connect(":memory:")
df.to_sql("perfume_prices", conn, if_exists="replace", index=False)

def sql(query):
    return pd.read_sql_query(query, conn)

# ============================================================
# STEP 2: Platform benchmark — avg price per platform per tier
# This is our "market rate" reference
# ============================================================

benchmarks = sql("""
    SELECT
        platform,
        price_tier,
        ROUND(AVG(price_inr), 0)    AS benchmark_price,
        ROUND(MIN(price_inr), 0)    AS min_price,
        ROUND(MAX(price_inr), 0)    AS max_price,
        COUNT(*)                    AS product_count
    FROM perfume_prices
    WHERE price_inr IS NOT NULL
      AND price_tier != 'Unknown'
    GROUP BY platform, price_tier
    ORDER BY platform, benchmark_price
""")

print("=== PLATFORM PRICE BENCHMARKS BY TIER ===")
print(benchmarks.to_string(index=False))

# ============================================================
# STEP 3: Price recommendation per product
# Logic:
# - If price < 80% of platform tier average → Underpriced → Increase
# - If price > 120% of platform tier average → Overpriced → Review
# - Otherwise → Competitively priced
# ============================================================

recommendations = sql("""
    WITH platform_tier_avg AS (
        SELECT
            platform,
            price_tier,
            AVG(price_inr) AS avg_price
        FROM perfume_prices
        WHERE price_inr IS NOT NULL
          AND price_tier != 'Unknown'
        GROUP BY platform, price_tier
    )
    SELECT
        p.platform,
        p.product_name,
        p.price_tier,
        ROUND(p.price_inr, 0)           AS current_price,
        ROUND(a.avg_price, 0)           AS tier_avg_price,
        ROUND(p.price_inr / a.avg_price * 100, 1) AS pct_of_avg,
        CASE
            WHEN p.price_inr < a.avg_price * 0.80
                THEN 'Underpriced — Consider Increase'
            WHEN p.price_inr > a.avg_price * 1.20
                THEN 'Overpriced — Review Positioning'
            ELSE 'Competitively Priced'
        END AS recommendation,
        CASE
            WHEN p.price_inr < a.avg_price * 0.80
                THEN ROUND(a.avg_price * 0.90, 0)
            WHEN p.price_inr > a.avg_price * 1.20
                THEN ROUND(a.avg_price * 1.10, 0)
            ELSE ROUND(p.price_inr, 0)
        END AS suggested_price
    FROM perfume_prices p
    JOIN platform_tier_avg a
      ON p.platform = a.platform
     AND p.price_tier = a.price_tier
    WHERE p.price_inr IS NOT NULL
      AND p.price_tier != 'Unknown'
    ORDER BY p.platform, recommendation, p.price_inr DESC
""")

print("\n=== PRICE RECOMMENDATIONS (sample) ===")
print(recommendations[[
    "platform", "product_name", "current_price",
    "tier_avg_price", "pct_of_avg", "recommendation", "suggested_price"
]].head(20).to_string(index=False))

# ============================================================
# STEP 4: Recommendation summary
# ============================================================

rec_summary = sql("""
    WITH platform_tier_avg AS (
        SELECT platform, price_tier, AVG(price_inr) AS avg_price
        FROM perfume_prices
        WHERE price_inr IS NOT NULL AND price_tier != 'Unknown'
        GROUP BY platform, price_tier
    )
    SELECT
        p.platform,
        CASE
            WHEN p.price_inr < a.avg_price * 0.80 THEN 'Underpriced'
            WHEN p.price_inr > a.avg_price * 1.20 THEN 'Overpriced'
            ELSE 'Competitive'
        END AS recommendation,
        COUNT(*) AS product_count,
        ROUND(AVG(ABS(p.price_inr - a.avg_price)), 0) AS avg_gap_inr
    FROM perfume_prices p
    JOIN platform_tier_avg a
      ON p.platform = a.platform
     AND p.price_tier = a.price_tier
    WHERE p.price_inr IS NOT NULL AND p.price_tier != 'Unknown'
    GROUP BY p.platform, recommendation
    ORDER BY p.platform, product_count DESC
""")

print("\n=== RECOMMENDATION SUMMARY BY PLATFORM ===")
print(rec_summary.to_string(index=False))

# ============================================================
# STEP 5: Visualise recommendations
# ============================================================

fig1 = px.bar(
    rec_summary,
    x="platform",
    y="product_count",
    color="recommendation",
    barmode="group",
    title="Price Recommendation Distribution by Platform",
    labels={"product_count": "Number of Products", "recommendation": "Recommendation"},
    color_discrete_map={
        "Underpriced":  "#2ecc71",
        "Competitive":  "#3498db",
        "Overpriced":   "#e74c3c"
    }
)
fig1.update_layout(plot_bgcolor="white")
fig1.show()

# ============================================================
# STEP 6: Underpriced products — biggest revenue uplift opportunity
# ============================================================

underpriced = sql("""
    WITH platform_tier_avg AS (
        SELECT platform, price_tier, AVG(price_inr) AS avg_price
        FROM perfume_prices
        WHERE price_inr IS NOT NULL AND price_tier != 'Unknown'
        GROUP BY platform, price_tier
    )
    SELECT
        p.platform,
        p.product_name,
        ROUND(p.price_inr, 0)               AS current_price,
        ROUND(a.avg_price * 0.90, 0)        AS suggested_price,
        ROUND(a.avg_price * 0.90
              - p.price_inr, 0)             AS potential_uplift_inr,
        ROUND((a.avg_price * 0.90
              - p.price_inr)
              / p.price_inr * 100, 1)       AS uplift_pct
    FROM perfume_prices p
    JOIN platform_tier_avg a
      ON p.platform = a.platform
     AND p.price_tier = a.price_tier
    WHERE p.price_inr < a.avg_price * 0.80
      AND p.price_inr IS NOT NULL
    ORDER BY potential_uplift_inr DESC
    LIMIT 15
""")

print("\n=== TOP 15 UNDERPRICED PRODUCTS — REVENUE UPLIFT OPPORTUNITIES ===")
print(underpriced.to_string(index=False))

fig2 = px.bar(
    underpriced,
    x="potential_uplift_inr",
    y="product_name",
    color="platform",
    orientation="h",
    title="Top 15 Underpriced Products — Potential Revenue Uplift (₹)",
    labels={
        "potential_uplift_inr": "Potential Uplift (₹)",
        "product_name": "Product"
    },
    color_discrete_map={
        "Amazon.in": "#FF9900",
        "Flipkart":  "#2874F0",
        "Nykaa":     "#FC2779"
    }
)
fig2.update_layout(
    yaxis={"categoryorder": "total ascending"},
    plot_bgcolor="white"
)
fig2.show()

# ============================================================
# STEP 7: Business summary for interview
# ============================================================

total_underpriced = rec_summary[rec_summary["recommendation"] == "Underpriced"]["product_count"].sum()
total_overpriced  = rec_summary[rec_summary["recommendation"] == "Overpriced"]["product_count"].sum()
total_competitive = rec_summary[rec_summary["recommendation"] == "Competitive"]["product_count"].sum()

print("\n" + "=" * 60)
print("DECISION LAYER SUMMARY — FOR INTERVIEW")
print("=" * 60)
print(f"Total products analysed  : {len(recommendations)}")
print(f"Underpriced (raise price): {total_underpriced} products")
print(f"Overpriced (review price): {total_overpriced} products")
print(f"Competitively priced     : {total_competitive} products")
if len(underpriced) > 0:
    print(f"Avg revenue uplift opportunity : ₹{underpriced['potential_uplift_inr'].mean():.0f} per underpriced product")
    print(f"Max single product uplift      : ₹{underpriced['potential_uplift_inr'].max():.0f}")

print(f"""
WHAT TO SAY IN INTERVIEW:
"I built a SQL-based price recommendation engine that benchmarks
each product against its platform and tier average. Out of {len(recommendations)}
products analysed, {total_underpriced} are underpriced relative to their
segment — representing a direct revenue uplift opportunity.
The engine flags {total_overpriced} overpriced products at risk of
losing conversions to cheaper alternatives."
""")

# Save recommendations to CSV
from google.colab import files
recommendations.to_csv("perfume_recommendations.csv", index=False, encoding="utf-8-sig")
files.download("perfume_recommendations.csv")

df_matches.to_csv("perfume_arbitrage.csv", index=False, encoding="utf-8-sig")
files.download("perfume_arbitrage.csv")

print("✅ Cell 14 complete. Saved perfume_recommendations.csv")

=== PLATFORM PRICE BENCHMARKS BY TIER ===
 platform price_tier  benchmark_price  min_price  max_price  product_count
Amazon.in     Budget           281.00      59.00     499.00            730
Amazon.in  Mid-range           916.00     500.00   1,999.00            302
Amazon.in    Premium         3,561.00   2,001.00   5,949.00             47
Amazon.in     Luxury        15,220.00   6,200.00  48,496.00             18
 Flipkart     Budget           278.00      57.00     499.00            593
 Flipkart  Mid-range           907.00     500.00   1,945.00            184
 Flipkart    Premium         3,132.00   2,145.00   4,392.00              4
 Flipkart     Luxury         7,045.00   6,290.00   7,800.00              2
    Nykaa     Budget           318.00      70.00     499.00             35
    Nykaa  Mid-range           993.00     500.00   1,900.00             61
    Nykaa    Premium         3,246.00   2,245.00   5,995.00             27
    Nykaa     Luxury        10,144.00   6,000.00  34,000.0


=== TOP 15 UNDERPRICED PRODUCTS — REVENUE UPLIFT OPPORTUNITIES ===
 platform                                                                                                                                                                           product_name  current_price  suggested_price  potential_uplift_inr  uplift_pct
Amazon.in                                                                                   Forest Essentials Intense Perfume Kesari | Lime Saffron & Oudh Blend Perfume | Lasts Through the Day       6,200.00        13,698.00              7,498.00      120.90
Amazon.in                                                                                                                                                                    Salvatore Ferragamo       6,213.00        13,698.00              7,485.00      120.50
Amazon.in                                                            Ralph Lauren - Polo Blue - Parfum - Men's Cologne - Aquatic & Fresh - With Citrus, Oak


DECISION LAYER SUMMARY — FOR INTERVIEW
Total products analysed  : 2040
Underpriced (raise price): 784 products
Overpriced (review price): 583 products
Competitively priced     : 673 products
Avg revenue uplift opportunity : ₹4961 per underpriced product
Max single product uplift      : ₹7498

WHAT TO SAY IN INTERVIEW:
"I built a SQL-based price recommendation engine that benchmarks
each product against its platform and tier average. Out of 2040
products analysed, 784 are underpriced relative to their
segment — representing a direct revenue uplift opportunity.
The engine flags 583 overpriced products at risk of
losing conversions to cheaper alternatives."



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Cell 14 complete. Saved perfume_recommendations.csv
